<div style="text-align: justify; max-width: 90ch">

# Lecture 02f: An external corpus at scale: Wikipedia

**Status: Optional / career-track.**

The course notes are 125 files the course controls. Real assistants also read documents from outside: public web pages, other teams' wikis, vendors' manuals.
Twelve English Wikipedia articles on the course's topics, fetched through the MediaWiki API, show what changes when documents come from outside:
the licence, polite fetching, the cost of scale, and two sources in one assistant.

**What this notebook covers**

- The licence decides what you may store and show.
- Fetching politely, with a cache.
- Chunk counts and embedding speed, and what they mean for thousands of pages.
- Two sources, two collections, and a question searched in both.

</div>

In [1]:
import os
import time
from pathlib import Path

import ollama
import pandas as pd
from dotenv import load_dotenv

from llm_course.corpus import fetch_wikipedia_pages, load_corpus
from llm_course.rag import LECTURE_KINDS, build_chroma_index, split_documents

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-02-rag"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. The licence first

Wikipedia's text is licensed [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/): you may copy, store and adapt it,
commercially too, if you **credit** the authors and **share** adaptations under the same licence. An assistant that quotes Wikipedia must say where the text came from.
The licence is a requirement for data exactly as it was for models in `lec_01b`; the course notes' own terms are in `corpus/LICENSE_NOTE.md`.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Fetching politely

`fetch_wikipedia_pages` calls the [MediaWiki API](https://www.mediawiki.org/wiki/API:Main_page) with `requests`:

- one page per request
- at most one request per second
- a User-Agent that says who is asking
- a pause-and-retry if the server answers "429 Too Many Requests"

Pages are cached in `data/wikipedia/`, so the second run makes no network call at all. A whole class fetching at once is exactly the traffic these rules exist for.

</div>

In [3]:
TITLES = [
    "Machine learning",
    "K-nearest neighbors algorithm",
    "Linear regression",
    "Logistic regression",
    "Naive Bayes classifier",
    "Support vector machine",
    "Cross-validation (statistics)",
    "Hyperparameter optimization",
    "K-means clustering",
    "Overfitting",
    "Leakage (machine learning)",
    "Regularization (mathematics)",
]

started = time.time()
wiki_docs = fetch_wikipedia_pages(TITLES, out_dir=DATA_DIR / "wikipedia")

print(f"{len(wiki_docs)} pages in {time.time() - started:.0f} s")

12 pages in 0 s


In [4]:
pd.DataFrame(
    [
        {"title": d.metadata["title"], "characters": len(d.page_content)}
        for d in wiki_docs
    ]
).sort_values("characters", ascending=False)

,title,characters
3,Logistic regression,238274
5,Support vector machine,100745
2,Linear regression,86534
4,Naive Bayes classifier,80829
11,Regularization (mathematics),69493
0,Machine learning,60265
8,K-means clustering,57420
6,Cross-validation (statistics),37614
1,K-nearest neighbors algorithm,34698
9,Overfitting,16101


In [5]:
print(wiki_docs[0].page_content[:400])

# Machine learning

Source: https://en.wikipedia.org/wiki/Machine_learning (revision 1378799175). Text by Wikipedia contributors, licensed CC BY-SA 4.0.

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly progr


<div style="text-align: justify; max-width: 90ch">

The attribution sits at the top of every cached file, and `fetch_wikipedia_pages` also puts the URL, revision and licence in each page's metadata,
which the splitter copies into every chunk: any chunk can be traced back to the exact version it came from, and credited.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Chunks and speed

The same splitter as the course notes (`lec_02c`), then a timing: how many chunks per second does the embedding model manage on *this* computer?

</div>

In [6]:
wiki_chunks = split_documents(wiki_docs, chunk_size=800, chunk_overlap=100)

print(len(wiki_chunks), "chunks from", len(wiki_docs), "pages")
print(wiki_chunks[10].metadata)

1404 chunks from 12 pages
{'source': 'wikipedia/Machine_learning.md', 'lecture': 0, 'kind': 'wikipedia', 'title': 'Machine learning', 'url': 'https://en.wikipedia.org/wiki/Machine_learning', 'revision': '1378799175', 'license': 'CC BY-SA 4.0', 'start_index': 4854, 'heading': 'Machine learning'}


In [7]:
client = ollama.Client(host=OLLAMA_HOST)

sample = [c.page_content for c in wiki_chunks[:128]]

started = time.time()
client.embed(model=EMBED_MODEL, input=sample)
per_second = len(sample) / (time.time() - started)

print(f"{per_second:.0f} chunks per second")

183 chunks per second


<div style="text-align: justify; max-width: 90ch">

Now scale it up. Each vector is 768 numbers of 4 bytes, about 3 KB per chunk before any index structures:

</div>

In [8]:
chunks_per_page = len(wiki_chunks) / len(wiki_docs)

rows = []

for label, pages in [
    ("these 12 pages", 12),
    ("1,000 pages", 1_000),
    ("100,000 pages", 100_000),
]:
    n_chunks = pages * chunks_per_page
    rows.append(
        {
            "corpus": label,
            "chunks": round(n_chunks),
            "embedding minutes": round(n_chunks / per_second / 60, 1),
            "vectors GB": round(n_chunks * 768 * 4 / 1e9, 2),
        }
    )

pd.DataFrame(rows)

,corpus,chunks,embedding minutes,vectors GB
0,these 12 pages,1404,0.1,0.00
1,"1,000 pages",117000,10.7,0.36
2,"100,000 pages",11700000,1066.8,35.94


<div style="text-align: justify; max-width: 90ch">

These twelve are long articles, so the estimate is on the high side, but the shape of the problem holds: at scale, embedding is a **batch job** measured in hours,
run once on strong hardware or a hosted embedding service, and changing the chunk size or the embedding model means paying for it again.
That is why the chunking and model decisions of `lec_02c` are worth measuring before the first large run.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. Two sources, one question

The Wikipedia chunks go into their own collection, next to the course notes, with the idempotent `build_chroma_index` of `lec_02d`.

⏱ **Skip if running long.**

The first run embeds both corpora: the Wikipedia pages here, and the course notes too if `lec_02d` has not built them yet.

</div>

In [9]:
wiki_store = build_chroma_index(
    wiki_chunks, "wikipedia", DATA_DIR / "chroma", EMBED_MODEL, OLLAMA_HOST
)

notes_store = build_chroma_index(
    split_documents(load_corpus(REPO / "corpus" / "uoa_py_course")),
    "course_notes",
    DATA_DIR / "chroma",
    EMBED_MODEL,
    OLLAMA_HOST,
)

Embedding 1404 chunks into 'wikipedia' (one-time)...


  256 / 1404


  512 / 1404


  768 / 1404


  1024 / 1404


  1280 / 1404


  1404 / 1404
Reusing 'course_notes': 2452 chunks in chroma/


In [10]:
question = "What is data leakage?"

rows = []

for name, store, search_filter in [
    ("course notes", notes_store, {"kind": {"$in": LECTURE_KINDS}}),
    ("Wikipedia", wiki_store, None),
]:
    for doc, distance in store.similarity_search_with_score(
        question, k=2, filter=search_filter
    ):
        rows.append(
            {
                "collection": name,
                "distance": round(distance, 3),
                "source": doc.metadata["source"],
                "text": doc.page_content[:70],
            }
        )

pd.DataFrame(rows)

,collection,distance,source,text
0,course notes,0.386,lecture_13/lec_13b_pipelines_preprocessing_mod...,## 2. The leakage problem\n\n**Data leakage** ...
1,course notes,0.411,lecture_13/lec_13g_automl_autogluon.md,- **Leakage is still your job.** AutoGluon pre...
2,Wikipedia,0.199,wikipedia/Leakage_machine_learning_.md,# Leakage (machine learning)\n\nSource: https:...
3,Wikipedia,0.243,wikipedia/Leakage_machine_learning_.md,=== Training example leakage ===\nRow-wise lea...


<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

Read the two chunks from each collection: does each one actually explain leakage? Where both do, which would you show a student of this course first, and why?
The notes use the course's own examples and variable names; Wikipedia is broader and more formal. Whatever you choose, the answer must say which source it used, and for Wikipedia, credit it.
Distances from two collections are comparable here only because both used the same embedding model.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- External text comes with a **licence**: store the attribution with every chunk (CC BY-SA requires credit and share-alike).
- Fetch **politely**: identify yourself, pace the requests, retry on 429, and cache so the second run is offline.
- Measure **chunks per second** on your own hardware before a large run; at scale, embedding is a batch job measured in hours.
- Separate sources can live in separate **collections**; distances compare only across collections built with the same embedding model.

**Next: lecture 3**, which measures the answers, not only the retrieval.

</div>